# Versuch SES: Stoffeigenschaften

| | |
|---|---|
| **Name** | *Vorname Nachname* |
| **Matrikelnummer** | *123456* |
| **Gruppe** | *…* |
| **Betreuer:in** | *…* |
| **Versuchstag** | *TT.MM.JJJJ* |

<br>

> **Notebook als Versuchsbericht:** Sie können dieses Notebook als Versuchsbericht abgeben. Werten Sie dazu alle Ihre Messdaten gemäß den Aufgaben aus und diskutieren Sie Ihre Ergebnisse in Markdown-Zellen an geeigneter Stelle im Notebook. Die Diskussion ist ein wichtiger Teil des Berichts. Laden Sie das Notebook als `.ipynb`-Datei bis zum nächsten Termin im moodle-Lernraum hoch. Gehören weitere Dateien dazu, zum Beispiel eine CSV-Datei, aus der Sie Messdaten eingelesen haben, oder Bilder, die nicht im Notebook eingebettet sind, laden Sie alles zusammen als `.zip`-Archiv hoch. Benennen Sie die Datei nach dem Schema `Gruppe_Nachname1_Nachname2_SES` mit der Bezeichnung Ihrer Gruppe (zum Beispiel CMo03) und den Nachnamen aller Gruppenmitglieder.

**Vorübergehend:** Im Praktikumsprofil fehlen derzeit noch Pakete, die dieses Notebook braucht. Führen Sie die folgende Zelle deshalb zu Beginn einmal aus, sie installiert die Pakete. Meldet ein späterer Import trotzdem einen Fehler, starten Sie den Kernel neu (**Kernel** → **Restart Kernel**) und führen Sie die Zellen erneut aus.

In [ ]:
%pip install --quiet numpy scipy matplotlib uncertainties odrpack==0.6.1

In diesem Notebook werden die Messdaten zum Versuch SES ausgewertet (Anleitung Kapitel 7, Aufgaben in den Abschnitten 7.1.5 und 7.2.5).

Es werden verwendet:

- `numpy` für Rechnungen mit Messwerten
- `matplotlib.pyplot` für Diagramme
- `odr_fit` aus `odrpack` für die linearen Fits mit Unsicherheiten in beide Richtungen

Die Auswertung besteht aus zwei Teilen:

1. Permittivität: Steigung, Kapazität und Ladungsanteile für vier Messreihen, relative Permittivität beider Plastikscheiben (Aufgaben 1a–1d)
2. Polarisation: Vergleich der vier Messreihen mit den theoretisch erwarteten Kurven (Aufgabe 2)

Notation wie im Versuch MEDA: $\bar x$ Mittelwert, $s_x$ Standardabweichung, $u_x$ Messunsicherheit. Ergebnisse werden in der Form $(\bar x \pm u_x)\,\mathrm{Einheit}$ angegeben. $u_x$ wird auf eine signifikante Stelle aufgerundet (auf zwei, wenn die erste Stelle eine 1 oder 2 ist), der Bestwert auf dieselbe Dezimalstelle gerundet.

Die verwendeten Python-Werkzeuge (Arrays, `plt.errorbar()`, `odr_fit`, Fehlerfortpflanzung) kennen Sie aus der Einführung PYT.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from odrpack import odr_fit

## 1. Permittivität

In diesem Teil wird aus der Ladespannung $U_1$ und der Spannung $U_2$ am Messkondensator die Kapazität $C_1$ für vier Messreihen bestimmt, daraus die relative Permittivität $\varepsilon_r$ der beiden Plastikscheiben.

Trägt man $U_2$ gegen $U_1$ auf, ist die Steigung der Ausgleichsgeraden (Bemerkung zu Aufgabe 1a):

$$
k = \frac{C_1}{C_1 + C_2}
$$

Daraus folgt die Kapazität (Gl. 7.4):

$$
C_1 = C_2 \cdot \frac{k}{1 - k}
$$

Zuerst werden die Messwerte definiert. Danach werden Steigungen, Kapazitäten, Ladungsanteile, Unsicherheiten und $\varepsilon_r$ berechnet.

### Messwerte und Konstanten definieren

Hier werden für die vier Messreihen (rote Plastikscheibe, Luft mit demselben Abstand, durchsichtige Plastikscheibe, Luft mit demselben Abstand) die Spannungen $U_1$ und $U_2$ definiert.

Vorgaben aus der Anleitung (Aufgaben 1a und 1c): $C_2 = 10\,\mathrm{nF}$ mit $u_{C_2} = 2\,\mathrm{nF}$. Die Multimeter haben eine Unsicherheit von $1\,\%$ des Messwerts:

$$
u_{U_1} = 0{,}01 \cdot U_1
$$

$$
u_{U_2} = 0{,}01 \cdot U_2
$$

In [ ]:
# Ihr Code hier


### Diagramm der Messwerte erstellen

Vor dem Fit werden die Messwerte dargestellt: $U_2$ gegen $U_1$ für alle vier Messreihen mit den Fehlerbalken $u_{U_1}$ und $u_{U_2}$. Prüfen Sie am Verlauf, ob der lineare Zusammenhang nach der Bemerkung zu Aufgabe 1a vorliegt.

Zur Erinnerung: `plt.errorbar(x, y, xerr=u_x, yerr=u_y, marker="o", linestyle="", label="Messwerte")` zeichnet die Messwerte mit Fehlerbalken. Jede Messreihe erhält ein eigenes `label`.

Achsen werden in der Form „Größe in Einheit“ beschriftet.

In [ ]:
# Ihr Code hier


### Aufgabe 1a: Steigung durch linearen Fit bestimmen

Die Ausgleichsgerade hat die Form:

$$
U_2 = k\,U_1 + b
$$

Weil beide Spannungen Unsicherheiten tragen, wird der Fit für jede Messreihe mit `odr_fit` durchgeführt. Er gewichtet die Messwerte mit ihren Unsicherheiten in beide Richtungen. Für die Unsicherheit von $k$ verlangt Aufgabe 1c die Methode minimaler und maximaler Steigung, sie folgt in einem späteren Schritt.

Zur Erinnerung: ODR berücksichtigt Unsicherheiten in $x$- und $y$-Richtung. Der Fit läuft in vier Schritten.

**1. Modellfunktion definieren.** Die Parameter stehen gemeinsam im Array `beta`. Die Funktion bekommt zuerst `x` und danach `beta`:

```python
def linear_odr(x, beta):
    return beta[0] * x + beta[1]
```

**2. Gewichte berechnen.** `odr_fit` erwartet die Unsicherheiten nicht direkt, sondern als Gewichte. Das Gewicht eines Messwerts ist eins geteilt durch das Quadrat seiner Unsicherheit:

$$
w_x = \frac{1}{u_x^2}
\qquad
w_y = \frac{1}{u_y^2}
$$

**3. Fit durchführen.** Übergeben werden die Modellfunktion, die Messwerte, die Startwerte `beta0` und die Gewichte:

```python
output = odr_fit(linear_odr, x, y, beta0=[1, 0], weight_x=1/u_x**2, weight_y=1/u_y**2)
```

**4. Ergebnisse auslesen.** Die Parameter stehen in `output.beta`. Ihre Unsicherheiten stehen wie bei `curve_fit` auf der Diagonalen der Kovarianzmatrix, hier heißt sie `output.cov_beta`:

```python
u_m = np.sqrt(output.cov_beta[0, 0])   # Steigung
u_b = np.sqrt(output.cov_beta[1, 1])   # Achsenabschnitt
```

**Wichtig:** `beta0` sind die Startwerte. `odr_fit` arbeitet mit relativen Schrittweiten und braucht sie in der richtigen Größenordnung, kein Startwert sollte genau null sein. `output.info` gleich 1 zeigt, dass der Fit konvergiert ist.

In [ ]:
# Ihr Code hier


### Aufgabe 1a: Kapazität berechnen

Aus der Steigung $k$ jeder Messreihe wird die Kapazität berechnet (Gl. 7.4):

$$
C_1 = C_2 \cdot \frac{k}{1 - k}
$$

Die Kapazitäten werden in $\mathrm{pF}$ ausgegeben.

In [ ]:
# Ihr Code hier


### Aufgabe 1b: Ladungsanteile berechnen

Aufgabe 1b fragt: Welcher Anteil der ursprünglich auf $C_1$ gespeicherten Ladung verbleibt nach dem Umschalten auf $C_1$, welcher Anteil fließt auf $C_2$?

Nach dem Hinweis zu Aufgabe 1b entspricht der auf $C_1$ verbleibende Anteil der Steigung $k$ aus Aufgabe 1a. Berechnen Sie beide Anteile für alle vier Messreihen in Prozent.

In [ ]:
# Ihr Code hier


### Aufgabe 1c: Minimale und maximale Steigung bestimmen

Nach dem Hinweis zu Aufgabe 1c wird die Unsicherheit von $k$ mit der Methode minimaler und maximaler Steigung bestimmt (Abschnitt 1.3.2). Dazu werden für jede Messreihe zwei Grenzgeraden durch den Messpunkt mit der kleinsten und den mit der größten Ladespannung $U_1$ gelegt (Indizes mit `np.argmin()` und `np.argmax()`):

- minimale Steigung durch $(U_{1,\mathrm{min}} - u_{U_1},\ U_2 + u_{U_2})$ und $(U_{1,\mathrm{max}} + u_{U_1},\ U_2 - u_{U_2})$,
- maximale Steigung durch $(U_{1,\mathrm{min}} + u_{U_1},\ U_2 - u_{U_2})$ und $(U_{1,\mathrm{max}} - u_{U_1},\ U_2 + u_{U_2})$.

Die Steigung einer Geraden durch zwei Punkte ist der Quotient aus $y$- und $x$-Differenz. Die größere Abweichung zur Steigung des Fits ist die Unsicherheit:

$$
u_k = \max\left( |k_\mathrm{max} - k|,\ |k - k_\mathrm{min}| \right)
$$

Dafür werden `max()` und `abs()` benutzt. Für das Diagramm im nächsten Schritt wird auch der Achsenabschnitt jeder Grenzgeraden berechnet: $b = y_1 - k\,x_1$ mit dem ersten Punkt $(x_1, y_1)$.

In [ ]:
# Ihr Code hier


### Diagramm mit Ausgleichsgeraden und Grenzgeraden erstellen

Für jede Messreihe werden die Fitgerade und die beiden Grenzgeraden in ein eigenes Diagramm der Messwerte eingezeichnet.

Zur Erinnerung: `plt.plot(x, linear_odr(x, output.beta), label="Fit")` zeichnet die Fitgerade an den $x$-Werten der Messung. Mit `linestyle="--"` werden die Grenzgeraden gestrichelt gezeichnet.

In [ ]:
# Ihr Code hier


### Aufgabe 1c: Unsicherheit der Kapazität berechnen

Die Unsicherheit der Kapazität wird nach Gl. 7.5 berechnet:

$$
\frac{u_{C_1}}{C_1} =
\sqrt{
\left(
\frac{u_k}{k\,(1 - k)}
\right)^2
+
\left(
\frac{u_{C_2}}{C_2}
\right)^2
}
$$

Setzen Sie für jede Messreihe $k$, $u_k$, $C_2$ und $u_{C_2}$ ein. Verwenden Sie für die Wurzel `np.sqrt()`.

**Hinweis:** Die Fehlerfortpflanzung kann auch mit dem Paket `uncertainties` berechnet werden. Dazu werden die Messgrößen zusammen mit ihren Unsicherheiten als Arrays angelegt, zum Beispiel `k = unp.uarray(k_red, u_k_red)` und `C_2 = unp.uarray(C2, u_C2)` (nach `import uncertainties.unumpy as unp`). Mit diesen Arrays wird wie gewohnt gerechnet, Funktionen wie Wurzel oder Kosinus werden dabei als `unp.sqrt()` und `unp.cos()` geschrieben. Bestwert und Unsicherheit des Ergebnisses liefern `unp.nominal_values(C_1)` und `unp.std_devs(C_1)`. Achten Sie auf die richtige Angabe signifikanter Stellen. `uncertainties` rundet Unsicherheiten kaufmännisch statt aufzurunden.

In [ ]:
# Ihr Code hier


### Aufgabe 1d: Relative Permittivität berechnen

Für jede Plastikscheibe wird $\varepsilon_r$ aus den Kapazitäten mit und ohne Scheibe bei gleichem Plattenabstand berechnet (Gl. 7.6):

$$
\varepsilon_r = \frac{C_{1,\mathrm{mit}}}{C_{1,\mathrm{ohne}}}
$$

Aufgabe 1d stellt dazu eine Denkaufgabe: Bei den Kapazitäten $C_1$ haben Sie die $20\,\%$ Toleranz des Messkondensators berücksichtigt. Muss diese Toleranz bei der Unsicherheit von $\varepsilon_r$ ebenfalls berücksichtigt werden? Begründen Sie Ihre Antwort in der Diskussion von Teil 1.

Die Unsicherheit wird nach Gl. 7.7 berechnet:

$$
\frac{u_{\varepsilon_r}}{\varepsilon_r} =
\sqrt{
\left(
\frac{u_{k,\mathrm{mit}}}{k_\mathrm{mit}\,(1 - k_\mathrm{mit})}
\right)^2
+
\left(
\frac{u_{k,\mathrm{ohne}}}{k_\mathrm{ohne}\,(1 - k_\mathrm{ohne})}
\right)^2
}
$$

$\varepsilon_r$ ist einheitenlos.

In [ ]:
# Ihr Code hier


> **Ergebnis Teil 1**
>
> Rote Scheibe: $k = \ldots \pm \ldots$, $C_1 = (\ldots \pm \ldots)\,\mathrm{pF}$, Ladungsanteil auf $C_1$: $(\ldots \pm \ldots)\,\%$, auf $C_2$: $(\ldots \pm \ldots)\,\%$
>
> Luft (Abstand der roten Scheibe): $k = \ldots \pm \ldots$, $C_1 = (\ldots \pm \ldots)\,\mathrm{pF}$, Ladungsanteil auf $C_1$: $(\ldots \pm \ldots)\,\%$, auf $C_2$: $(\ldots \pm \ldots)\,\%$
>
> Durchsichtige Scheibe: $k = \ldots \pm \ldots$, $C_1 = (\ldots \pm \ldots)\,\mathrm{pF}$, Ladungsanteil auf $C_1$: $(\ldots \pm \ldots)\,\%$, auf $C_2$: $(\ldots \pm \ldots)\,\%$
>
> Luft (Abstand der durchsichtigen Scheibe): $k = \ldots \pm \ldots$, $C_1 = (\ldots \pm \ldots)\,\mathrm{pF}$, Ladungsanteil auf $C_1$: $(\ldots \pm \ldots)\,\%$, auf $C_2$: $(\ldots \pm \ldots)\,\%$
>
> $\varepsilon_{r,\mathrm{rot}} = \ldots \pm \ldots$, $\varepsilon_{r,\mathrm{durchsichtig}} = \ldots \pm \ldots$

### Diskussion Teil 1

*…*

## 2. Polarisation

In diesem Teil werden die vier Messreihen der Intensität $I$ in Abhängigkeit vom Winkel $\psi$ mit den theoretisch erwarteten Kurven aus Abschnitt 7.2.2 verglichen.

Aufgabe 2 verlangt, die Messungen aus allen vier Versuchsteilen jeweils graphisch darzustellen und die theoretisch erwarteten Kurven mit einzuzeichnen.

Gesetz von Malus (Gl. 7.8):

$$
I = I_0 \cos^2\psi
$$

Drei Polfilter (Gl. 7.9). Da die Winkelskala des mittleren Filters willkürlich ist, legen Sie $\psi = 0°$ in eines der gemessenen Minima:

$$
I = \frac{I_0}{4} \sin^2(2\psi)
$$

Zirkular polarisiertes Licht, konstante Intensität:

$$
I = \frac{1}{2}\,I_0
$$

Elliptisch polarisiertes Licht (Gl. 7.10) mit dem von Ihnen eingestellten Winkel $\alpha = 45° \pm \Delta$. Welches Vorzeichen gilt, erkennen Sie an der Lage des Maximums (Aufgabe 2):

$$
I = I_0 \left( \cos^2\alpha \, \cos^2(\psi - \alpha) + \sin^2\alpha \, \sin^2(\psi - \alpha) \right)
$$

Diskutieren Sie, inwiefern Messung und Theorie übereinstimmen. Berücksichtigen Sie dabei, dass die Theoriekurven ideale Filter annehmen: Reale Polarisationsfilter und $\lambda/4$-Plättchen absorbieren auch in Durchlassrichtung einen Teil des Lichts.

Zum dritten Polfilter fragt die Durchführung (Abschnitt 7.2.4) außerdem: Was beobachten Sie, und warum?

Die Auftragung entsteht während der Durchführung von Hand auf dem Polpapier (Abschnitt 7.2.4). Ihre Antworten gehören in die Diskussion von Teil 2.

### Diskussion Teil 2

*…*

## 3. Zusammenfassung der Ergebnisse

Zum Schluss werden die wichtigsten Ergebnisse des gesamten Versuchs gesammelt ausgegeben.

Die Zusammenfassung enthält für alle vier Messreihen die Steigung, die Kapazität und die Ladungsanteile aus Aufgabe 1b, dazu die relative Permittivität beider Plastikscheiben:

$$
k \pm u_k
\qquad
C_1 \pm u_{C_1}
\qquad
\varepsilon_r \pm u_{\varepsilon_r}
$$

Achten Sie darauf, alle Ergebnisse mit passenden Einheiten auszugeben. Tragen Sie die Ergebnisse in die Ergebnis-Zelle darunter ein und achten Sie dabei auf die richtige Angabe signifikanter Stellen.

In [ ]:
# Ihr Code hier


> **Ergebnis (Zusammenfassung)**
>
> Rote Scheibe: $k = \ldots \pm \ldots$, $C_1 = (\ldots \pm \ldots)\,\mathrm{pF}$, Ladungsanteil auf $C_1$: $(\ldots \pm \ldots)\,\%$, auf $C_2$: $(\ldots \pm \ldots)\,\%$
>
> Luft (Abstand der roten Scheibe): $k = \ldots \pm \ldots$, $C_1 = (\ldots \pm \ldots)\,\mathrm{pF}$, Ladungsanteil auf $C_1$: $(\ldots \pm \ldots)\,\%$, auf $C_2$: $(\ldots \pm \ldots)\,\%$
>
> Durchsichtige Scheibe: $k = \ldots \pm \ldots$, $C_1 = (\ldots \pm \ldots)\,\mathrm{pF}$, Ladungsanteil auf $C_1$: $(\ldots \pm \ldots)\,\%$, auf $C_2$: $(\ldots \pm \ldots)\,\%$
>
> Luft (Abstand der durchsichtigen Scheibe): $k = \ldots \pm \ldots$, $C_1 = (\ldots \pm \ldots)\,\mathrm{pF}$, Ladungsanteil auf $C_1$: $(\ldots \pm \ldots)\,\%$, auf $C_2$: $(\ldots \pm \ldots)\,\%$
>
> $\varepsilon_{r,\mathrm{rot}} = \ldots \pm \ldots$, $\varepsilon_{r,\mathrm{durchsichtig}} = \ldots \pm \ldots$
>
> …

## 4. Diskussion

*…*

## Abgabe als Versuchsbericht

Wenn Sie dieses Notebook als Versuchsbericht abgeben, prüfen Sie vorher, dass alle Aufgaben ausgewertet und Ihre Ergebnisse in Markdown-Zellen diskutiert sind, und füllen Sie die Kopfzelle aus. Laden Sie das Notebook als `.ipynb`-Datei bis zum nächsten Termin im moodle-Lernraum hoch. Gehören weitere Dateien dazu, zum Beispiel eine CSV-Datei, aus der Sie Messdaten eingelesen haben, oder Bilder, die nicht im Notebook eingebettet sind, laden Sie das Notebook zusammen mit diesen Dateien als `.zip`-Archiv hoch. Benennen Sie die Datei nach dem Schema `Gruppe_Nachname1_Nachname2_SES` mit der Bezeichnung Ihrer Gruppe (zum Beispiel CMo03) und den Nachnamen aller Gruppenmitglieder.